# Volatility Dynamics 14 — Seven horizons, four distribution models and an augmented Markov lift

**Status: preregistered, historical exploratory, not executed.** The historical 2024–2026 source has been inspected in prior modules; this does **not** produce untouched confirmation.

**Seven horizons:** 5, 10, 15, 30, 60, 120, 240 minutes. **Five models:** M1 Gaussian two-EMA, M2 Student-t, M3 constant broad-tail Gaussian mixture, M4 phase-conditioned broad-tail mixture, and AM two-state-proxy Markovian lift with propagated variance and five-minute shock mechanism.

Read `discussions/Volatility_Dynamics/031_chatgpt_vd14_markov_lift_preregistration.md` before interpreting the experiment. The rare Gaussian mixture component is **not a labelled physical discontinuity**. The augmented Markov model uses observable EMA proxies (not an identified hidden state or exact jump diffusion).


## 0. Environment, parameters and synthetic tests

In [1]:
from pathlib import Path
import os, sys, json, time, warnings
import numpy as np
import pandas as pd
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').exists():
    ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').exists(), 'Run the notebook within deribit-market-data'
load_dotenv(ROOT/'.env');load_dotenv()
if str(ROOT/'src') not in sys.path:sys.path.insert(0,str(ROOT/'src'))
from data_engineering.modeling import markov_lift_distributions as ml
print('VD14',ml.VERSION)
print(json.dumps(ml.registration(),indent=2))
OUT=ROOT/'notebooks'/'Volatility_Dynamics'/'14'
OUT.mkdir(parents=True,exist_ok=True)
HIST=Path(os.environ['DERIBIT_DATA_ROOT'])/'history'/'price_5m'
ASSETS=tuple(a.strip().upper() for a in os.getenv('VD14_ASSETS','AVAX,BTC,ETH,HYPE,SOL,TRX,XRP').split(',') if a.strip())
BOOT=int(os.getenv('VD14_BOOT',str(ml.BOOT)))
assert BOOT>=100, 'For diagnostic runs set VD14_BOOT>=100; normal final run 1000'
print('History:',HIST,'| assets:',ASSETS,'| bootstrap:',BOOT)
START=time.monotonic()


VD14 VD14-0.1.0
{
  "version": "VD14-0.1.0",
  "horizons": [
    5,
    10,
    15,
    30,
    60,
    120,
    240
  ],
  "primary_horizon_min": 60,
  "bar_min": 5,
  "fast_halflife_min": 30,
  "slow_halflife_min": 240,
  "train_fraction": 0.6,
  "validation_fraction": 0.2,
  "min_history_bars": 288,
  "min_history_coverage": 0.8,
  "student_df_grid": [
    4.0,
    6.0,
    10.0,
    20.0,
    40.0
  ],
  "shock_scale_grid": [
    3.0,
    5.0,
    8.0
  ],
  "tail_quantile": 0.95,
  "historical_window_exclusive_end": "2026-09-19 00:00:00+00:00",
  "historical_test_is_exploratory": true,
  "note": "Fixed historical period was previously inspected in VD12/VD13; not fresh confirmation."
}
History: D:\MarketData\Deribit\history\price_5m | assets: ('AVAX', 'BTC', 'ETH', 'HYPE', 'SOL', 'TRX', 'XRP') | bootstrap: 1000


## 1. Raw five-minute spot closes, independent causal features and targets

In [2]:
raw,excluded=ml.load_returns(HIST,ASSETS)
print('Exclusions:',excluded,'| raw rows:',len(raw))
panel=ml.panelize(raw)
del raw
features=ml.make_features(panel)
frame=ml.make_targets(panel,features)
print('Prepared in',round(time.monotonic()-START,1),'s')
print(panel.groupby('split',sort=False).agg(days=('date','nunique'),rows=('return_5m','size'),valid_returns=('return_5m','count')).to_string())
print('Eligible origins before target censoring:',frame.groupby('split').ready.sum().to_dict())
print('First and last bars:',str(panel.state_time.min()),'->',str(panel.state_time.max()))
assert tuple(ml.HORIZONS)==(5,10,15,30,60,120,240)
assert set(frame.split.unique())=={'train','valid','test'}


Exclusions: {'HYPE': 'insufficient history'} | raw rows: 1292538
Prepared in 9.9 s
       days    rows  valid_returns
split                             
train   448  774138         774132
valid   150  259200         259200
test    150  259200         259200
Eligible origins before target censoring: {'test': 259200, 'train': 772410, 'valid': 259200}
First and last bars: 2024-09-01 00:05:00+00:00 -> 2026-09-18 23:55:00+00:00


## 2. Fit five distributions at each horizon and bootstrap in UTC-day blocks

The one-step AM shock mechanism and bivariate state transition are fitted from train data (with validation selection of the 5m wide-component scale). Longer-horizon AM forecasts propagate these fitted states. M1–M4 fit separately at each horizon. **Origin-level predictions are discarded after scoring each horizon**, keeping output small and memory bounded.


In [3]:
scores,boot,parameters=ml.run_all(frame,B=BOOT)
print('Models:',sorted(scores.model.unique().tolist()))
print('Horizon coverage:')
print(scores.groupby('horizon_min').agg(n_models=('model','nunique'),origins=('n_origins','min'),days=('n_days','min')).to_string())
assert len(scores)==len(ml.MODELS)*len(ml.HORIZONS)
assert (scores.groupby('horizon_min').n_origins.nunique()==1).all()
print('Scored in',round(time.monotonic()-START,1),'s')


Models: ['AM_markov_lift', 'M1_gaussian', 'M2_student_t', 'M3_constant_mix', 'M4_conditional_mix']
Horizon coverage:
             n_models  origins  days
horizon_min                         
5                   5   259194   150
10                  5   259188   150
15                  5   259182   150
30                  5   259164   150
60                  5   259128   150
120                 5   259056   150
240                 5   258912   150
Scored in 503.0 s


## 3. All model metrics; 60-minute primary in detail

In [4]:
cols=['horizon_min','model','nll','rv_log_mse','tail_logloss','tail_brier','tail_observed','tail_predicted','interval_miss95']
display(scores[cols].round(6))
display(boot[(boot.horizon_min==ml.PRIMARY)&(boot.metric.isin(['nll','tail_logloss']))].round(6))
# Calibration is not optional: all five models predict physical absolute-return tails.
print('Two-sided nominal interval miss rate = 0.05; observed values outside 0.05 indicate calibration differences.')


,horizon_min,model,nll,rv_log_mse,tail_logloss,tail_brier,tail_observed,tail_predicted,interval_miss95
0,5,M1_gaussian,-3.775897,33.916570,0.130961,0.017043,0.018048,0.009102,0.232154
1,5,M2_student_t,-5.081550,33.916570,0.081984,0.016794,0.018048,0.009748,0.231525
2,5,M3_constant_mix,-4.987685,33.916570,0.088292,0.016895,0.018048,0.009445,0.248771
3,5,M4_conditional_mix,-4.989089,33.916570,0.088948,0.016931,0.018048,0.009579,0.248551
4,5,AM_markov_lift,-5.340947,34.267107,0.078895,0.016791,0.018048,0.012169,0.120265
5,10,M1_gaussian,-4.876741,12.050431,0.103426,0.016789,0.018122,0.010506,0.094194
6,10,M2_student_t,-5.053414,12.050431,0.078475,0.016687,0.018122,0.012217,0.089996
7,10,M3_constant_mix,-5.040389,12.050431,0.078225,0.016731,0.018122,0.011328,0.104006
8,10,M4_conditional_mix,-5.040427,12.050431,0.078145,0.016730,0.018122,0.011238,0.103871
9,10,AM_markov_lift,-5.007702,12.117515,0.082709,0.016798,0.018122,0.012498,0.105213


,horizon_min,comparison,metric,delta,q025,q975,test_days,origins
112,60,M2_student_t - M1_gaussian,nll,-0.107388,-0.142206,-0.079069,150,259128
114,60,M2_student_t - M1_gaussian,tail_logloss,-0.016282,-0.022254,-0.010982,150,259128
116,60,M3_constant_mix - M1_gaussian,nll,-0.098253,-0.130625,-0.072573,150,259128
118,60,M3_constant_mix - M1_gaussian,tail_logloss,-0.016208,-0.022850,-0.010453,150,259128
120,60,M3_constant_mix - M2_student_t,nll,0.009135,0.006870,0.012085,150,259128
122,60,M3_constant_mix - M2_student_t,tail_logloss,0.000074,-0.000320,0.000461,150,259128
124,60,M4_conditional_mix - M3_constant_mix,nll,-0.000437,-0.000603,-0.000267,150,259128
126,60,M4_conditional_mix - M3_constant_mix,tail_logloss,-0.000005,-0.000043,0.000032,150,259128
128,60,AM_markov_lift - M4_conditional_mix,nll,0.143047,0.107228,0.191002,150,259128
130,60,AM_markov_lift - M4_conditional_mix,tail_logloss,0.027575,0.019164,0.037031,150,259128


Two-sided nominal interval miss rate = 0.05; observed values outside 0.05 indicate calibration differences.


## 4. Predeclared summary (not a tradability or prospective-confirmation gate)

In [5]:
result=ml.decision(scores,boot)
print(json.dumps(result,indent=2))
assert result['status'] in ('HISTORICAL_EXPLORATORY','INSUFFICIENT_DATA')
assert not result['not_prospective'] is False


{
  "status": "HISTORICAL_EXPLORATORY",
  "n_test_days": 150,
  "primary_horizon_min": 60,
  "flags": {
    "t_better_than_gaussian": true,
    "constant_mix_better_than_gaussian": true,
    "constant_mix_better_than_t": false,
    "conditional_mix_better_than_constant": true,
    "markov_better_than_conditional": false,
    "markov_better_than_t": false
  },
  "strong_markov_candidate": false,
  "not_prospective": true,
  "interpretation": "Historical 2024-2026 data was previously examined. All positive flags require fresh prospective confirmation."
}


## 5. Save only compact reproducible artifacts

In [6]:
scores.to_csv(OUT/'14_scores.csv',index=False,float_format='%.9g')
boot.to_csv(OUT/'14_day_bootstrap.csv',index=False,float_format='%.9g')
(OUT/'14_decision.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
manifest={'module':'Volatility_Dynamics_14','version':ml.VERSION,
  'registration':ml.registration(),'source_dir':str(HIST),
  'assets':sorted(panel.underlying.unique()),'excluded_assets':excluded,
  'earliest_date':str(panel.state_time.min()),'latest_date':str(panel.state_time.max()),
  'day_splits':{str(k):int(v) for k,v in panel.groupby('split').date.nunique().items()},
  'bootstrap_draws':BOOT,'models':list(ml.MODELS),
  'parameter_fits':parameters,'decision':result,
  'scientific_caveat':'Observational Markov-proxy transitions and statistical tail mixtures; previous data exposure prevents prospective inference.'}
(OUT/'14_run_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('Saved 4 compact artifacts under',OUT)
print('No raw origin predictions, return tensors or market data were saved. No database writes.')


Saved 4 compact artifacts under D:\Projects\deribit_market_data\notebooks\Volatility_Dynamics\14
No raw origin predictions, return tensors or market data were saved. No database writes.
